In [1]:
from useful.helpers import *
%matplotlib tk
seed = 42
jax.config.update("jax_enable_x64", True)
key = jax.random.PRNGKey(seed)

wigner_function_from_inference, t, f = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")

wigner_function_from_inference = wigner_function_from_inference
f = f

In [2]:
n = 10
m, k = len(f), len(t)  # or len(white_noise_stress)

white_noise_matrices = np.empty((n, m, k))
white_noise_matrices_smoothed = np.empty((n, m, k))

for i in range(n):
    white_noise = np.random.standard_normal(len(f))
    white_noise_stress, _, _ = Stress_re(white_noise, time=t, supress_print=True)
    white_noise_stress = white_noise_stress.real
    print(f"Calculated {i}th white noise stress")

    white_noise_matrices[i] = white_noise_stress
    white_noise_matrices_smoothed[i] = smooth_matrix(white_noise_stress, smoothing_lvl=5, mode="gaussian")

# print("Pickling 1...")
# pickle_me_this("bunch_of_white_noise", white_noise_matrices)
# print("Pickling 2...")
# pickle_me_this("bunch_of_white_noise_smoothed", white_noise_matrices_smoothed)

Calculated 0th white noise stress
Calculated 1th white noise stress
Calculated 2th white noise stress
Calculated 3th white noise stress
Calculated 4th white noise stress
Calculated 5th white noise stress
Calculated 6th white noise stress
Calculated 7th white noise stress
Calculated 8th white noise stress
Calculated 9th white noise stress


In [ ]:
# white_noise_matrices = unpickle_me_this("bunch_of_white_noise.pickle")[:10]  # (30)
# white_noise_matrices_smoothed = unpickle_me_this("bunch_of_white_noise_smoothed.pickle.pickle")[:10] # (30)

In [ ]:
prior_snr = 3440/1980 # stress ratio of: S(1.40, 65)/S(0.3420, -1340.1)
prior_snr

In [52]:
smooth_wigner = smooth_matrix(wigner_function_from_inference, smoothing_lvl=5, mode="gaussian")
# alpha = np.std(smooth_wigner) / np.std(white_noise_matrices_smoothed[0])
# alpha = 1e5
alpha = 1.83e3 / np.max(white_noise_matrices_smoothed[0])

# print("Scaling by sigma_wigner ", np.std(smooth_wigner), " divided by sigma noise ", np.std(white_noise_matrices_smoothed[0]), " = ", alpha)
#

In [53]:
# W_sn_list = [smooth_wigner - 1e4 * W_n_i_smooth for W_n_i_smooth in white_noise_matrices_smoothed]
W_sn_list = smooth_wigner[None, :, :] - alpha * white_noise_matrices_smoothed

In [54]:
for w in W_sn_list[:4]:
    visualize_stress(w, rows=f, cols=t, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [58]:
# smoothed_wigner_averaged_over_noise_background = np.mean(np.array(W_sn_list), axis=0)
smoothed_wigner_averaged_over_noise_background =  W_sn_list.mean(axis=0)

In [56]:
# double_smoothed_wigner_averaged_over_noise_background = smooth_matrix(smoothed_wigner_averaged_over_noise_background, smoothing_lvl=15, mode="gaussian")
# double_smoothed_wigner_averaged_over_noise_background[np.where(double_smoothed_wigner_averaged_over_noise_background<0)] = 0

visualize_stress(smoothed_wigner_averaged_over_noise_background, rows=f, cols=t, smooth=False)
# visualize_stress(smooth_wigner, rows=f, cols=t, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [ ]:
after_snr = 3350 / 1960
after_snr

In [ ]:
print(np.sum(smoothed_wigner_averaged_over_noise_background.real))
print(np.sum(smooth_wigner.real))